# Suffix direct-MH SMC tail estimation

This notebook runs only `suffix_direct_mh_smc` and estimates $P_{p_M}(\phi \geq a)$ from each run's final weighted SMC population. The estimator is self-normalized importance sampling: it is consistent, but generally biased for a finite number of particles.

In [ ]:
# 1. Mount Google Drive for persistent experiment outputs.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone the repository, or safely fast-forward an existing clean checkout.
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/FrancescoVinciguerra/rea_of_llms_supplementary_code-main.git'
REPO_ROOT = Path('/content/rea_of_llms_supplementary_code-main')

if (REPO_ROOT / '.git').is_dir():
    dirty = subprocess.run(
        ['git', '-C', str(REPO_ROOT), 'status', '--porcelain'],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    if dirty:
        raise RuntimeError('The existing Colab checkout has local changes; review them before updating.')
    subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only', 'origin', 'main'], check=True)
elif REPO_ROOT.exists():
    raise RuntimeError(f'{REPO_ROOT} exists but is not a Git repository.')
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)
print('Repository:', REPO_ROOT)

In [ ]:
# 3. Enter and validate the repository root.
import os
os.chdir(REPO_ROOT)
assert (REPO_ROOT / 'suffix_balanced_smc_experiment' / 'run_tail_estimation.py').is_file()
assert (REPO_ROOT / 'requirements-colab.txt').is_file()
print('cwd =', Path.cwd())

In [ ]:
# 4. Install the repository dependencies in this Colab runtime.
%pip install -q -r requirements-colab.txt

In [ ]:
# 5. Inspect PyTorch, CUDA, and the selected Colab accelerator.
import torch
print('torch:', torch.__version__)
print('compiled CUDA:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# 6. EDIT PARAMETERS HERE. This is the only cell that configures the experiment.
SEEDS = [200, 201, 202]
NUM_PARTICLES = 256
MAX_NEW_TOKENS = 100
LAMBDAS = '0,-0.25,-0.5,-0.75,-1,-1.5,-2'
MCMC_STEPS_PER_LEVEL = 5
ESS_THRESHOLD = 0.5

# Historical MASK_* notebook names configure suffix lengths, not token masks.
MASK_MODE = 'stochastic'
MASK_SIZES = '1,2,4,8,16,32,64,100'
MASK_RHO_SCHEDULE = '0.20,0.18,0.16,0.14,0.12,0.10,0.06,0.04'

PROPOSAL_TEMPERATURE = 1.0
BATCH_SIZE = 32
TAIL_THRESHOLDS = '6,8,10,12,15,20'
EXPERIMENT_LABEL = 'suffix_smc_tail_estimation'
OUTPUT_ROOT = Path('/content/drive/MyDrive/rea_suffix_smc_tail_outputs')
DEVICE = 'cuda'
SKIP_COMPLETED_RUNS = True

assert PROPOSAL_TEMPERATURE == 1.0
assert len(SEEDS) == len(set(SEEDS))
print('Seeds:', SEEDS)
print('Output:', OUTPUT_ROOT / EXPERIMENT_LABEL)

In [ ]:
# 7. Run independent seeds. Progress is reported once per completed SMC level.
import sys

if DEVICE == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError('DEVICE=cuda but this Colab runtime has no CUDA GPU.')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
command = [
    sys.executable, 'suffix_balanced_smc_experiment/run_tail_estimation.py',
    '--device', DEVICE,
    '--seeds', ','.join(map(str, SEEDS)),
    '--num-particles', str(NUM_PARTICLES),
    '--max-new-tokens', str(MAX_NEW_TOKENS),
    '--lambdas', LAMBDAS,
    '--mcmc-steps-per-level', str(MCMC_STEPS_PER_LEVEL),
    '--ess-threshold', str(ESS_THRESHOLD),
    '--mask-mode', MASK_MODE,
    '--mask-sizes', MASK_SIZES,
    '--mask-rho-schedule', MASK_RHO_SCHEDULE,
    '--proposal-temperature', str(PROPOSAL_TEMPERATURE),
    '--batch-size', str(BATCH_SIZE),
    '--tail-thresholds', TAIL_THRESHOLDS,
    '--experiment-label', EXPERIMENT_LABEL,
    '--output-root', str(OUTPUT_ROOT),
]
if SKIP_COMPLETED_RUNS:
    command.append('--skip-completed')
subprocess.run(command, check=True)

In [ ]:
# 8. Load the selected independent runs, show compact tables, and draw one figure.
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

payloads = []
for seed in SEEDS:
    run_path = OUTPUT_ROOT / EXPERIMENT_LABEL / f'seed_{seed}' / 'run.json'
    if not run_path.is_file():
        raise FileNotFoundError(run_path)
    payload = json.loads(run_path.read_text(encoding='utf-8'))
    assert payload['algorithm'] == 'suffix_direct_mh_smc'
    assert payload['seed'] == seed
    payloads.append(payload)

run_rows = []
tail_rows = []
for payload in payloads:
    summary = payload['summary']
    run_rows.append({
        'seed': payload['seed'],
        'runtime_seconds': summary['runtime_seconds'],
        'lambda_final': summary['lambda_final'],
        'num_particles': summary['num_particles'],
        'mean_phi': summary['mean_phi'],
        'max_phi': summary['max_phi'],
        'smc_ess_over_n': summary['smc_ess_over_n'],
        'corrected_importance_ess_over_n': summary['corrected_importance_ess_over_n'],
    })
    for row in summary['tails']:
        tail_rows.append({'seed': payload['seed'], **row})

runs_df = pd.DataFrame(run_rows)
tails_df = pd.DataFrame(tail_rows)
display(runs_df)
aggregate_df = tails_df.groupby('threshold', as_index=False).agg(
    mean_raw_count=('raw_count', 'mean'),
    std_raw_count=('raw_count', 'std'),
    mean_raw_tilted_fraction=('raw_tilted_fraction', 'mean'),
    std_raw_tilted_fraction=('raw_tilted_fraction', 'std'),
    mean_probability_under_p_m=('estimated_probability_under_p_m', 'mean'),
    std_probability_under_p_m=('estimated_probability_under_p_m', 'std'),
).fillna(0.0)
display(aggregate_df)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for payload in payloads:
    phi = np.asarray([sample['phi'] for sample in payload['samples']], dtype=float)
    axes[0].hist(phi, bins=24, alpha=0.35, label=f"seed {payload['seed']}")
axes[0].set(title='Final SMC particle phi', xlabel='phi', ylabel='raw particle count')
axes[0].legend()

x = aggregate_df['threshold'].to_numpy(dtype=float)
mean = aggregate_df['mean_probability_under_p_m'].to_numpy(dtype=float)
std = aggregate_df['std_probability_under_p_m'].to_numpy(dtype=float)
axes[1].plot(x, mean, marker='o', label='independent-seed mean')
if len(payloads) > 1:
    axes[1].fill_between(x, np.clip(mean - std, 0, 1), np.clip(mean + std, 0, 1), alpha=0.2, label='+/- 1 seed SD')
axes[1].set(title='Estimated base-model tail', xlabel='threshold', ylabel='estimated P under p_M', ylim=(0, 1))
axes[1].legend()
fig.tight_layout()
plt.show()